In [25]:
import os
import glob

import numpy as np
import polars as pl
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve, auc

In [2]:
WORKING_DIR = '/group/pmc021/amunif/epi-thesis/workflow/08_HepG2/'

In [4]:
# Check CUDA device
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


In [6]:
# Load dataset
gene_pl = pl.read_parquet(os.path.join(WORKING_DIR, 'dataset', 'gene_w_label_value_1.parquet'))
gene_pl

gene_id,H3K4me3,H3K4me3_count,H3K9ac,H3K9ac_count,H3K9me3,H3K9me3_count,H3K27ac,H3K27ac_count,H3K27me3,H3K27me3_count,value_1,label
str,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,f64,i32
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0888452,0
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,4.04743,1
"""XLOC_000008""","[0.0, 0.0, … 0.0]",6,"[0.0, 0.0, … 0.0]",3,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",2,"[0.0, 0.0, … 0.0]",0,26.7934,1
…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0


In [10]:
# Load permutation list
permutations = pl.read_parquet('permutation.parquet')['markers_perm'].to_list()
len(permutations)

325

In [12]:
item = permutations[100]
item_name = '-'.join(item)
print(item)
print(item_name)

['H3K9me3', 'H3K9ac', 'H3K27ac', 'H3K4me3']
H3K9me3-H3K9ac-H3K27ac-H3K4me3


In [13]:
# Create dataframe based on histone marker permutation item
gene_perm_pl = gene_pl.with_columns(
    pl.struct(item).map_elements(
        lambda x: [x[col_name] for col_name in item],
        return_dtype = pl.List(pl.List(pl.Float64))
    )
    .alias('histone')
)

In [15]:
# Select X and y column
X = gene_perm_pl.select(pl.col('histone')).to_series().to_list()
y = gene_perm_pl.select(pl.col('label')).to_series().to_list()

In [16]:
# Convert to Numpy array
X = np.array(X)
y = np.array(y)

In [17]:
# Split the dataset into training, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.666, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

In [18]:
print("Train set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)

Train set shape: (7399, 4, 4000) (7399,)
Validation set shape: (7377, 4, 4000) (7377,)
Test set shape: (7378, 4, 4000) (7378,)


In [19]:
# Convert numpy arrays to PyTorch tensors
X_train = torch.from_numpy(X_train).float().unsqueeze(1).to(device)
X_val = torch.from_numpy(X_val).float().unsqueeze(1).to(device)
X_test = torch.from_numpy(X_test).float().unsqueeze(1).to(device)

y_train = torch.from_numpy(y_train).float().to(device)
y_val = torch.from_numpy(y_val).float().to(device)
y_test = torch.from_numpy(y_test).float().to(device)

In [20]:
# Create DataLoaders
batch_size = 32

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [21]:
# Define DeepClassifer Class
class DeepClassifier(nn.Module):
    def __init__(self, input_dim = [5, 4000], out_channels = 50, conv_kernel_size = 10, max_pooling_kernel_size = 5):
        super(DeepClassifier, self).__init__()
        
        flatten_dim = out_channels * ((input_dim[1] - conv_kernel_size + 1) // max_pooling_kernel_size)

        self.conv = nn.Conv2d(1, out_channels, kernel_size=(input_dim[0], conv_kernel_size))
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d((1, max_pooling_kernel_size))
        self.dropout = nn.Dropout(0.5)
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(flatten_dim, 625)
        self.linear2 = nn.Linear(625, 125)
        self.linear3 = nn.Linear(125, 2)
        self.softmax = nn.LogSoftmax(dim=1)
        

    def forward(self, x):
        # Stage 1: filter bank -> squashing -> max pooling
        x = self.conv(x)
        x = self.relu(x)
        x = self.pool(x)

        # Stage 2: standar 2-layer neural network        
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        x = self.relu(x)
        x = self.linear3(x)
        x = self.softmax(x)
        return x

In [22]:
# Find the feature's number of row from histone permutation item
n_rows_item = len(item)
print(n_rows_item)

4


In [28]:
# Initialize the model
model = DeepClassifier(input_dim = [n_rows_item, 4000]).to(device)
criterion = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001)
num_epochs = 100

# For tracking train and validation
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []
train_aucs, val_aucs = [], []

# For saving the best model
best_val_metric = float('-inf')
best_model_path = os.path.join(WORKING_DIR, "experiments", "model", f'{item_name}.pth')

In [52]:
# Preparing the output dataframe
output_df = pd.DataFrame(columns=[
    "item_name",
    "train_loss_min", "train_loss_avg", "train_loss_max",
    "train_acc_min", "train_acc_avg", "train_acc_max",
    "train_auc_min", "train_auc_avg", "train_auc_max",
    "val_loss_min", "val_loss_avg", "val_loss_max",
    "val_acc_min", "val_acc_avg", "val_acc_max",
    "val_auc_min", "val_auc_avg", "val_auc_max",
])

In [30]:
# Run the training and validation phase
for epoch in range(num_epochs):
    # Training phase
    model.train()  # Set the model to training mode
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    train_true_labels = []
    train_predicted_probs = []
    
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.cuda(), labels.type(torch.LongTensor).cuda())
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

        train_true_labels.extend(labels.cpu().numpy())
        train_predicted_probs.extend(predicted.cpu().numpy())
    
    avg_train_loss = train_loss / len(train_loader)
    train_accuracy = 100 * train_correct / train_total
    train_auc_score = roc_auc_score(train_true_labels, train_predicted_probs)
    
    train_losses.append(round(avg_train_loss, 4))
    train_accuracies.append(round(train_accuracy, 2))
    train_aucs.append(round(train_auc_score, 2))
    
    # Validation phase
    model.eval()  # Set the model to evaluation mode
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    val_metric = 0
    val_true_labels = []
    val_predicted_probs = []
    
    with torch.no_grad():  # Disable gradient computation
        for inputs, labels in val_loader:
            outputs = model(inputs)
            loss = criterion(outputs.cuda(), labels.type(torch.LongTensor).cuda())
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            val_true_labels.extend(labels.cpu().numpy())
            val_predicted_probs.extend(predicted.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = 100 * val_correct / val_total
    val_metric = val_correct / len(val_loader)

    # Save the best model
    if val_metric > best_val_metric:
        best_val_metric = val_metric
        torch.save(model.state_dict(), best_model_path)
        print(f"New best model saved with validation metric: {best_val_metric:.4f}")
    
    val_true_labels = np.array(val_true_labels)
    val_predicted_probs = np.array(val_predicted_probs)
    val_auc_score = roc_auc_score(val_true_labels, val_predicted_probs)

    val_losses.append(round(avg_val_loss, 4))
    val_accuracies.append(round(val_accuracy, 2))
    val_aucs.append(round(val_auc_score, 2))
    
    print(f'Epoch [{epoch+1}/{num_epochs}] - '
          f'[TRAIN] Loss: {avg_train_loss:.4f}, Accuracy: {train_accuracy:.2f}%, AUC: {train_auc_score:.2f} - '
          f'[VAL] Loss: {avg_val_loss:.4f}, Accuracy: {val_accuracy:.2f}%, AUC: {val_auc_score:.2f}')
print("Training finished!")

New best model saved with validation metric: 19.4242
Epoch [1/100] - [TRAIN] Loss: 0.6457, Accuracy: 61.21%, AUC: 0.53 - [VAL] Loss: 0.6005, Accuracy: 60.82%, AUC: 0.50
Epoch [2/100] - [TRAIN] Loss: 0.5837, Accuracy: 60.82%, AUC: 0.50 - [VAL] Loss: 0.5662, Accuracy: 60.82%, AUC: 0.50
New best model saved with validation metric: 24.7749
Epoch [3/100] - [TRAIN] Loss: 0.5595, Accuracy: 66.41%, AUC: 0.58 - [VAL] Loss: 0.5478, Accuracy: 77.58%, AUC: 0.76
New best model saved with validation metric: 24.9827
Epoch [4/100] - [TRAIN] Loss: 0.5462, Accuracy: 78.09%, AUC: 0.76 - [VAL] Loss: 0.5372, Accuracy: 78.23%, AUC: 0.77
New best model saved with validation metric: 25.0303
Epoch [5/100] - [TRAIN] Loss: 0.5376, Accuracy: 78.50%, AUC: 0.77 - [VAL] Loss: 0.5315, Accuracy: 78.38%, AUC: 0.78
Epoch [6/100] - [TRAIN] Loss: 0.5342, Accuracy: 78.61%, AUC: 0.78 - [VAL] Loss: 0.5274, Accuracy: 78.11%, AUC: 0.78
Epoch [7/100] - [TRAIN] Loss: 0.5287, Accuracy: 78.47%, AUC: 0.78 - [VAL] Loss: 0.5239, Accu

In [33]:
# Load the best model
best_model = DeepClassifier(input_dim = [n_rows_item, 4000]).to(device)
best_model.load_state_dict(torch.load(best_model_path))
best_model.eval()

# Predict on test set
test_predictions = []
test_true_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = best_model(inputs)
        # probabilities = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs.data, 1)
        test_predictions.extend(predicted.cpu().numpy())
        test_true_labels.extend(labels.cpu().numpy())

# Calculate evaluation metric
test_acc_score = round(accuracy_score(test_true_labels, test_predictions) * 100, 2)
print(f"Accuracy Score on test set: {test_acc_score:.2f} %")
test_auc_score = round(roc_auc_score(test_true_labels, test_predictions), 2)
print(f"AUC Score on test set: {test_auc_score:.2f}")

Accuracy Score on test set: 79.24 %
AUC Score on test set: 0.79


In [34]:
# Create dataframe from array
experiment_results = {
    'train_loss': train_losses,
    'train_accuracy': train_accuracies,
    'train_auc': train_aucs,
    'val_loss': val_losses,
    'val_accuracy': val_accuracies,
    'val_auc': val_aucs
}

In [64]:
experiment_pl = pl.DataFrame(experiment_results)

In [65]:
experiment_pl.write_csv(os.path.join(WORKING_DIR, 'experiments', 'details', f'{item_name}.csv'))

In [37]:
def min_avg_max(lst):
    return round(min(lst), 2), round(sum(lst)/len(lst), 2), round(max(lst), 2)

In [38]:
train_loss_min, train_loss_avg, train_loss_max = min_avg_max(train_losses)
train_acc_min, train_acc_avg, train_acc_max = min_avg_max(train_accuracies)
train_auc_min, train_auc_avg, train_auc_max = min_avg_max(train_aucs)

print(f"{train_loss_min:.2f}, {train_loss_avg:.2f}, {train_loss_max:.2f}")
print(f"{train_acc_min}, {train_acc_avg}, {train_acc_max}")
print(f"{train_auc_min}, {train_auc_avg}, {train_auc_max}")

0.47, 0.49, 0.65
60.82, 78.54, 79.75
0.5, 0.78, 0.79


In [39]:
val_loss_min, val_loss_avg, val_loss_max = min_avg_max(val_losses)
val_acc_min, val_acc_avg, val_acc_max = min_avg_max(val_accuracies)
val_auc_min, val_auc_avg, val_auc_max = min_avg_max(val_aucs)

print(f"{val_loss_min:.2f}, {val_loss_avg:.2f}, {val_loss_max:.2f}")
print(f"{val_acc_min}, {val_acc_avg}, {val_acc_max}")
print(f"{val_auc_min}, {val_auc_avg}, {val_auc_max}")

0.50, 0.50, 0.60
60.82, 78.13, 79.0
0.5, 0.77, 0.78


In [40]:
print(f"{test_acc_score}, {test_auc_score}")

79.24, 0.79


In [53]:
# Prepare the output file
output_rows = []

output_rows.append([
    item_name,
    train_loss_min, train_loss_avg, train_loss_max,
    train_acc_min, train_acc_avg, train_acc_max,
    train_auc_min, train_auc_avg, train_auc_max,
    val_loss_min, val_loss_avg, val_loss_max,
    val_acc_min, val_acc_avg, val_acc_max,
    val_auc_min, val_auc_avg, val_auc_max,
    
])

output_rows_df = pd.DataFrame(output_rows, 
    columns=[
        "item_name",
        "train_loss_min", "train_loss_avg", "train_loss_max",
        "train_acc_min", "train_acc_avg", "train_acc_max",
        "train_auc_min", "train_auc_avg", "train_auc_max",
        "val_loss_min", "val_loss_avg", "val_loss_max",
        "val_acc_min", "val_acc_avg", "val_acc_max",
        "val_auc_min", "val_auc_avg", "val_auc_max",
    ]
)

output_df = pd.concat([output_df, output_rows_df], ignore_index=True)

output_df


/tmp/ipykernel_387165/3815267995.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  output_df = pd.concat([output_df, output_rows_df], ignore_index=True)


,item_name,train_loss_min,train_loss_avg,train_loss_max,train_acc_min,train_acc_avg,train_acc_max,train_auc_min,train_auc_avg,train_auc_max,val_loss_min,val_loss_avg,val_loss_max,val_acc_min,val_acc_avg,val_acc_max,val_auc_min,val_auc_avg,val_auc_max
0,H3K9me3-H3K9ac-H3K27ac-H3K4me3,0.47,0.49,0.65,60.82,78.54,79.75,0.5,0.78,0.79,0.5,0.5,0.6,60.82,78.13,79.0,0.5,0.77,0.78


In [54]:
start = 0
end = 50

output_df.to_csv(os.path.join(WORKING_DIR, "experiments", "min-avg-max", 
                              f"batch-{str(start + 1).rjust(3, '0')}-{str(end).rjust(3, '0')}.csv"), 
                 header=True, index=False)

In [71]:
# Get CSV files list from a folder
csv_files = glob.glob(os.path.join(WORKING_DIR, "experiments", "min-avg-max", "*.csv"))
print(csv_files)

['/group/pmc021/amunif/epi-thesis/workflow/08_HepG2/experiments/min-avg-max/batch-301-350.csv', '/group/pmc021/amunif/epi-thesis/workflow/08_HepG2/experiments/min-avg-max/batch-001-050.csv', '/group/pmc021/amunif/epi-thesis/workflow/08_HepG2/experiments/min-avg-max/batch-051-100.csv', '/group/pmc021/amunif/epi-thesis/workflow/08_HepG2/experiments/min-avg-max/batch-151-200.csv', '/group/pmc021/amunif/epi-thesis/workflow/08_HepG2/experiments/min-avg-max/batch-101-150.csv', '/group/pmc021/amunif/epi-thesis/workflow/08_HepG2/experiments/min-avg-max/batch-201-250.csv', '/group/pmc021/amunif/epi-thesis/workflow/08_HepG2/experiments/min-avg-max/batch-251-300.csv']


In [72]:
# Read each CSV file into DataFrame
# This creates a list of dataframes
df_list = (pd.read_csv(file) for file in csv_files)

In [73]:
# Concatenate all DataFrames
big_df = pd.concat(df_list, ignore_index=True)

In [74]:
big_df.to_csv(os.path.join(WORKING_DIR, "experiments", "all_results.csv"), header=True, index=False)

In [66]:
def min_avg_max(lst):
    return round(min(lst), 2), round(sum(lst)/len(lst), 2), round(max(lst), 2)

In [70]:
start = 0
end = 1

output_rows = []

for item in permutations[start:end]:
    # Generate item name
    item_name = '-'.join(item)
    print(f"Item: {item_name}")

    # Create dataframe based on histone marker permutation item
    gene_perm_pl = gene_pl.with_columns(
        pl.struct(item).map_elements(
            lambda x: [x[col_name] for col_name in item],
            return_dtype = pl.List(pl.List(pl.Float64))
        )
        .alias('histone')
    )

    # Select X and y column
    X = gene_perm_pl.select(pl.col('histone')).to_series().to_list()
    y = gene_perm_pl.select(pl.col('label')).to_series().to_list()

    # Convert to Numpy array
    X = np.array(X)
    y = np.array(y)

    # Split the dataset into training, validation, and test sets
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.666, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
    print("Train set shape:", X_train.shape, y_train.shape)
    print("Validation set shape:", X_val.shape, y_val.shape)
    print("Test set shape:", X_test.shape, y_test.shape)

    # Convert numpy arrays to PyTorch tensors
    X_train = torch.from_numpy(X_train).float().unsqueeze(1).to(device)
    X_val = torch.from_numpy(X_val).float().unsqueeze(1).to(device)
    X_test = torch.from_numpy(X_test).float().unsqueeze(1).to(device)

    y_train = torch.from_numpy(y_train).float().to(device)
    y_val = torch.from_numpy(y_val).float().to(device)
    y_test = torch.from_numpy(y_test).float().to(device)

    # Create DataLoaders
    batch_size = 32

    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    val_dataset = TensorDataset(X_val, y_val)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    test_dataset = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    # Initialize the model
    model = DeepClassifier(input_dim = [len(item), 4000]).to(device)
    criterion = nn.NLLLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001)
    num_epochs = 10

    # For tracking train and validation
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []
    train_aucs, val_aucs = [], []

    # Preparing the output dataframe
    output_df = pd.DataFrame(columns=[
        "item_name",
        "train_loss_min", "train_loss_avg", "train_loss_max",
        "train_acc_min", "train_acc_avg", "train_acc_max",
        "train_auc_min", "train_auc_avg", "train_auc_max",
        "val_loss_min", "val_loss_avg", "val_loss_max",
        "val_acc_min", "val_acc_avg", "val_acc_max",
        "val_auc_min", "val_auc_avg", "val_auc_max",
    ])

    # For saving the best model
    best_val_metric = float('-inf')
    best_model_path = os.path.join(WORKING_DIR, "experiments", "model", f'{item_name}.pth')

    # Run the training and validation phase
    for epoch in range(num_epochs):
        # Training phase
        model.train()  # Set the model to training mode
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        train_true_labels = []
        train_predicted_probs = []
        
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs.cuda(), labels.type(torch.LongTensor).cuda())
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

            train_true_labels.extend(labels.cpu().numpy())
            train_predicted_probs.extend(predicted.cpu().numpy())
        
        avg_train_loss = train_loss / len(train_loader)
        train_accuracy = 100 * train_correct / train_total
        train_auc_score = roc_auc_score(train_true_labels, train_predicted_probs)
        
        train_losses.append(round(avg_train_loss, 4))
        train_accuracies.append(round(train_accuracy, 2))
        train_aucs.append(round(train_auc_score, 2))
        
        # Validation phase
        model.eval()  # Set the model to evaluation mode
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        val_metric = 0
        val_true_labels = []
        val_predicted_probs = []
        
        with torch.no_grad():  # Disable gradient computation
            for inputs, labels in val_loader:
                outputs = model(inputs)
                loss = criterion(outputs.cuda(), labels.type(torch.LongTensor).cuda())
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

                val_true_labels.extend(labels.cpu().numpy())
                val_predicted_probs.extend(predicted.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = 100 * val_correct / val_total
        val_metric = val_correct / len(val_loader)

        # Save the best model
        if val_metric > best_val_metric:
            best_val_metric = val_metric
            torch.save(model.state_dict(), best_model_path)
            print(f"New best model saved with validation metric: {best_val_metric:.4f}")
        
        val_true_labels = np.array(val_true_labels)
        val_predicted_probs = np.array(val_predicted_probs)
        val_auc_score = roc_auc_score(val_true_labels, val_predicted_probs)

        val_losses.append(round(avg_val_loss, 4))
        val_accuracies.append(round(val_accuracy, 2))
        val_aucs.append(round(val_auc_score, 2))
        
        print(f'Epoch [{epoch+1}/{num_epochs}] - '
            f'[TRAIN] Loss: {avg_train_loss:.4f}, Accuracy: {train_accuracy:.2f}%, AUC: {train_auc_score:.2f} - '
            f'[VAL] Loss: {avg_val_loss:.4f}, Accuracy: {val_accuracy:.2f}%, AUC: {val_auc_score:.2f}')
    print("Training finished!")

    # Load the best model
    best_model = DeepClassifier(input_dim = [len(item), 4000]).to(device)
    best_model.load_state_dict(torch.load(best_model_path))
    best_model.eval()

    # Predict on test set
    test_predictions = []
    test_true_labels = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = best_model(inputs)
            # probabilities = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs.data, 1)
            test_predictions.extend(predicted.cpu().numpy())
            test_true_labels.extend(labels.cpu().numpy())

    # Calculate evaluation metric
    test_acc_score = round(accuracy_score(test_true_labels, test_predictions) * 100, 2)
    print(f"Accuracy Score on test set: {test_acc_score:.2f} %")
    test_auc_score = round(roc_auc_score(test_true_labels, test_predictions), 2)
    print(f"AUC Score on test set: {test_auc_score:.2f}")

    # Create dataframe from array
    experiment_results = {
        'train_loss': train_losses,
        'train_accuracy': train_accuracies,
        'train_auc': train_aucs,
        'val_loss': val_losses,
        'val_accuracy': val_accuracies,
        'val_auc': val_aucs
    }

    # Save the experiment results
    experiment_pl = pl.DataFrame(experiment_results)
    experiment_pl.write_csv(os.path.join(WORKING_DIR, 'experiments', 'details', f'{item_name}.csv'))

    # Find the min, avg, max for training and validation step
    train_loss_min, train_loss_avg, train_loss_max = min_avg_max(train_losses)
    train_acc_min, train_acc_avg, train_acc_max = min_avg_max(train_accuracies)
    train_auc_min, train_auc_avg, train_auc_max = min_avg_max(train_aucs)

    val_loss_min, val_loss_avg, val_loss_max = min_avg_max(val_losses)
    val_acc_min, val_acc_avg, val_acc_max = min_avg_max(val_accuracies)
    val_auc_min, val_auc_avg, val_auc_max = min_avg_max(val_aucs)

    output_rows.append([
        item_name,
        train_loss_min, train_loss_avg, train_loss_max,
        train_acc_min, train_acc_avg, train_acc_max,
        train_auc_min, train_auc_avg, train_auc_max,
        val_loss_min, val_loss_avg, val_loss_max,
        val_acc_min, val_acc_avg, val_acc_max,
        val_auc_min, val_auc_avg, val_auc_max,
        
    ])

# Saving current batch results
output_rows_df = pd.DataFrame(output_rows, 
    columns=[
        "item_name",
        "train_loss_min", "train_loss_avg", "train_loss_max",
        "train_acc_min", "train_acc_avg", "train_acc_max",
        "train_auc_min", "train_auc_avg", "train_auc_max",
        "val_loss_min", "val_loss_avg", "val_loss_max",
        "val_acc_min", "val_acc_avg", "val_acc_max",
        "val_auc_min", "val_auc_avg", "val_auc_max",
    ]
)

output_rows_df.to_csv(os.path.join(WORKING_DIR, "experiments", "min-avg-max", 
                              f"batch-{str(start + 1).rjust(3, '0')}-{str(end).rjust(3, '0')}.csv"), 
                 header=True, index=False)

print("Batch FINISHED!!!")

Item: H3K4me3
Train set shape: (7399, 1, 4000) (7399,)
Validation set shape: (7377, 1, 4000) (7377,)
Test set shape: (7378, 1, 4000) (7378,)
New best model saved with validation metric: 19.4242
Epoch [1/10] - [TRAIN] Loss: 0.6315, Accuracy: 60.82%, AUC: 0.50 - [VAL] Loss: 0.5934, Accuracy: 60.82%, AUC: 0.50
New best model saved with validation metric: 24.8095
Epoch [2/10] - [TRAIN] Loss: 0.5749, Accuracy: 64.68%, AUC: 0.56 - [VAL] Loss: 0.5524, Accuracy: 77.69%, AUC: 0.76
Epoch [3/10] - [TRAIN] Loss: 0.5468, Accuracy: 77.88%, AUC: 0.76 - [VAL] Loss: 0.5357, Accuracy: 77.58%, AUC: 0.77
Epoch [4/10] - [TRAIN] Loss: 0.5349, Accuracy: 77.71%, AUC: 0.77 - [VAL] Loss: 0.5298, Accuracy: 77.55%, AUC: 0.77
New best model saved with validation metric: 24.8312
Epoch [5/10] - [TRAIN] Loss: 0.5266, Accuracy: 77.77%, AUC: 0.77 - [VAL] Loss: 0.5250, Accuracy: 77.76%, AUC: 0.77
Epoch [6/10] - [TRAIN] Loss: 0.5269, Accuracy: 77.85%, AUC: 0.77 - [VAL] Loss: 0.5225, Accuracy: 77.70%, AUC: 0.77
Epoch [7/1